# Lista de Exercício 7
### Introdução à Visão Computacional (SEL0339/SEL5886)

**Instruções:**

 1. Esta lista consiste de 2 exercícios.
 1. Deve-se colocar comentários nos códigos desenvolvidos.
 1. As perguntas devem ser respondidas também como comentários no arquivo.
 1. Colocar seu nome e número USP abaixo.
 1. Quaisquer problemas na execução das listas, entrar em contato com os monitores.
 1. Depois de terminado os exercícios, deve ser gerado um arquivo **extensão .ipynb** para ser enviado ao professor pelo E-DISCIPLINAS da disciplina até a data máxima de entrega.
 1. Caso não seja enviado, o aluno ficará sem nota.


---


 <table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/LAVI-USP/SEL0339-SEL5886_2024/blob/main/praticas/Lista_de_Exercicio_7.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Executar no Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/LAVI-USP/SEL0339-SEL5886_2024/blob/main/praticas/Lista_de_Exercicio_7.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />Ver codigo fonte no GitHub</a>
  </td>
</table>

`Nome: Marcus Vinícius Costa Reis`

`Número USP: 12549384`

### 1 (5.0) - Métricas de Avaliação

Utilize os modelos que foram treinados nas listas anteriores e faça as análises no conjunto de teste das métricas de validação.

Os modelos treinados na Lista 5 - Ex. 4) Modelos Classificação  e os modelos da Lista 6 - Ex. 2.1 (CNN) e 2.2 (Modelos pré-treinados)

Obs: **provavelmente os modelos não foram salvos com as seeds etc, então voces vão precisar 'treinar novamente' e não tem problema caso os valores tenham alguma variação dos valores originais postados na Lista 5 e 6.**

Análises que precisam ser feitas.

* Tabela de Contigência
* Sensibilidade (Recall)
* Especificidade (Precision)
* Acurácia
* Matriz de confusão
* F1-score
* ROC-AUC e a curva ROC (apenas para o conjunto de dados de classificação binária)

Podem utilizar bibliotecas do sklearn que já tenham essas implementados, não precisa implementar nada do zero. Avaliem o modelo SEMPRE no conjunto de teste para tirar as conclusões finais!

Para cada modelo que será feito essas analises faça uma breve discussão dos resultados (qual a métrica seria a melhor para aquele problema de classificação, qual seria pior etc, apenas uma análise crítica)

### Fórmula do F$\beta$-Score

Existe uma métrica em que ponderamos a Precision e Recall.

$$
F_{\beta} = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{(\beta^2 \cdot \text{Precision}) + \text{Recall}}
$$

#### Onde:
- $\beta > 1$: Dá mais peso ao Recall.
- $\beta < 1$: Dá mais peso à Precisão.
- $\beta = 1$: É equivalente ao F1-score.

### Interpretação:
- Permite personalizar o balanço entre Precisão e Recall.
- Útil em problemas específicos, como:
  - **Prioridade para Recall**: Segurança, detecção de fraudes, etc.
  - **Prioridade para Precisão**: Recomendações, etc.

### Fórmula do F1-Score

$$
F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

#### Onde:
- $\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$
- $\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$

### Interpretação:
- Um valor de $F_1$ próximo de 1 indica bom equilíbrio entre precisão e recall.
- É usado para avaliar modelos em cenários com dados desbalanceados.


In [2]:
!pip install ucimlrepo

In [13]:
## SEU CODIGO AQUI

# Treinando novamente os modelos da Lista 5 ==============================================================================

from ucimlrepo import fetch_ucirepo

# Importando outras bibliotecas que serão utilizadas
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

# Importando os classificadores
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# carregando o conjunto de dados
iris = fetch_ucirepo(id=53)

# atributos
X_iris = iris.data.features
# variavel resposta
y_iris = iris.data.targets
# Ajustando os valores da variável resposta y_iris por meio da aplicação do
# One-Hot Encoding com pd.get_dummies
y_iris_encoded = pd.get_dummies(y_iris)

# Normalizando os conjuntos de dados Iris e Heart Disease com StandardScaler
scaler = StandardScaler()                  # Instanciando objeto do StandardScaler
X_iris_std = scaler.fit_transform(X_iris)  # Aplicando em X_iris

# Realizando a divisão de treino/teste para ambos os datasets
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(X_iris_std, y_iris_encoded, test_size=0.3, random_state=42)

# Iris dataset
model_rf = RandomForestClassifier().fit(X_iris_train, y_iris_train)  # RandomForestClassifier
model_knn = KNeighborsClassifier().fit(X_iris_train, y_iris_train)   # KNeighborsClassifier
model_xgb = XGBClassifier().fit(X_iris_train, y_iris_train)          # XGBClassifier

# Treinando novamente o modelo CNN do exercicio 6

import tensorflow as tf
import tensorflow_datasets as tfds

# Criando pipeline de entrada

(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

# Criando pipeline de treinamento

def normalize_img(image, label):
  """Normalizes images: `uint8` -> `float32`."""
  return tf.cast(image, tf.float32) / 255., label

ds_train = ds_train.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(128)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

# Criando pipeline de avaliação

ds_test = ds_test.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.batch(128)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

# Modelos (complexidade será aumentada mudando-se a quantidade de filtros e neurônios)

# CNN com uma camada de convolução com 4 filtros e uma camada densa com 32 neurônios

modelo1 = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(4, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

modelo1.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

modelo1.fit(
    ds_train,
    epochs=6,
    validation_data=ds_test,
)


Dl Completed...:   0%|          | 0/5 [00:00<?, ? file/s]

Dataset mnist downloaded and prepared to /root/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.


/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 29s 47ms/step - loss: 0.9792 - sparse_categorical_accuracy: 0.7170 - val_loss: 0.2224 - val_sparse_categorical_accuracy: 0.9366
Epoch 2/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 32s 41ms/step - loss: 0.2049 - sparse_categorical_accuracy: 0.9398 - val_loss: 0.1583 - val_sparse_categorical_accuracy: 0.9526
Epoch 3/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 14s 29ms/step - loss: 0.1486 - sparse_categorical_accuracy: 0.9575 - val_loss: 0.1242 - val_sparse_categorical_accuracy: 0.9616
Epoch 4/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - loss: 0.1176 - sparse_categorical_accuracy: 0.9654 - val_loss: 0.1082 - val_sparse_categorical_accuracy: 0.9659
Epoch 5/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 12s 26ms/step - loss: 0.0966 - sparse_categorical_accuracy: 0.9712 - val_loss: 0.0931 - val_sparse_categorical_accuracy: 0.9695
Epoch 6/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 0.0830 - sparse_categorical_accuracy: 0.9748 - val_loss: 0.0831 - val_sparse_categorical_accuracy: 0.973

In [18]:
# MÉTRICAS DE AVALIAÇÃO ================================================================================================

from sklearn.metrics import confusion_matrix, classification_report

# Convertendo os rótulos verdadeiros para um formato numérico
y_iris_test_orig = y_iris_test.idxmax(axis=1).astype(str)  # Rótulos verdadeiros como strings
models = {'Random Forest': model_rf, 'KNN': model_knn, 'XGBoost': model_xgb}

for name, model in models.items():
    print(f"\n### Resultados para o modelo: {name} ###")

    # Predição do modelo (evitando o uso de argmax diretamente)
    y_pred = model.predict(X_iris_test)  # Previsão no formato one-hot
    y_pred_orig = pd.DataFrame(y_pred, columns=y_iris_test.columns).idxmax(axis=1).astype(str)

    # Geração da matriz de confusão
    cm = confusion_matrix(y_iris_test_orig, y_pred_orig, labels=y_iris_test.columns)
    print(f"Matriz de Confusão:\n{cm}")

    # Métricas
    report = classification_report(y_iris_test_orig, y_pred_orig, output_dict=True)
    recall = report["macro avg"]["recall"]
    precision = report["macro avg"]["precision"]
    f1 = report["macro avg"]["f1-score"]
    accuracy = report["accuracy"]

    print(f"Sensibilidade (Recall): {recall:.2f}")
    print(f"Especificidade (Precision): {precision:.2f}")
    print(f"Acurácia: {accuracy:.2f}")
    print(f"F1-Score: {f1:.2f}")

# Predição com o modelo CNN no conjunto de teste

print("\n### Resultados para o modelo: CNN ###")

y_true = []  # Lista para armazenar os rótulos verdadeiros
y_pred = []  # Lista para armazenar os rótulos preditos

for images, labels in ds_test:
    # Rótulos verdadeiros
    y_true.extend(labels.numpy())
    # Rótulos preditos
    predictions = modelo1.predict(images, verbose=0)  # Saída no formato probabilístico
    y_pred.extend(np.argmax(predictions, axis=1))  # Obter o índice da classe predita

# Convertendo listas para arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculando a Matriz de Confusão
cm = confusion_matrix(y_true, y_pred)
print("Matriz de Confusão:")
print(cm)

# Relatório de classificação e outras métricas
report = classification_report(y_true, y_pred, output_dict=True)
recall = report["macro avg"]["recall"]
precision = report["macro avg"]["precision"]
f1 = report["macro avg"]["f1-score"]
accuracy = report["accuracy"]

print(f"\nMétricas para o modelo CNN:")
print(f"Sensibilidade (Recall): {recall:.2f}")
print(f"Especificidade (Precision): {precision:.2f}")
print(f"Acurácia: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")



### Resultados para o modelo: Random Forest ###
Matriz de Confusão:
[[19  0  0]
 [ 0 13  0]
 [ 0  0 13]]
Sensibilidade (Recall): 1.00
Especificidade (Precision): 1.00
Acurácia: 1.00
F1-Score: 1.00

### Resultados para o modelo: KNN ###
Matriz de Confusão:
[[19  0  0]
 [ 0 13  0]
 [ 0  0 13]]
Sensibilidade (Recall): 1.00
Especificidade (Precision): 1.00
Acurácia: 1.00
F1-Score: 1.00

### Resultados para o modelo: XGBoost ###
Matriz de Confusão:
[[19  0  0]
 [ 0 13  0]
 [ 0  0 13]]
Sensibilidade (Recall): 1.00
Especificidade (Precision): 1.00
Acurácia: 1.00
F1-Score: 1.00

### Resultados para o modelo: CNN ###
Matriz de Confusão:
[[ 971    0    2    0    0    0    3    2    2    0]
 [   0 1127    3    1    0    1    3    0    0    0]
 [   3    2 1002    6    4    0    2    6    6    1]
 [   0    0    4  991    0    3    0    3    6    3]
 [   2    0    2    0  965    0    2    3    2    6]
 [   3    0    0   10    1  867    7    0    2    2]
 [   5    4    1    0    3    4  939    0    

# 2 - Análise de modelo treinado

Neste exemplos vamos analisar 2x modelos já treinados em um conjunto de dados ficticios de predição de uma doença, em que a classe 1 é a doença, não precisa se preocupar o sigfinicado das variveis V1, V2 etc.

Baixar o .csv

https://drive.google.com/drive/folders/1KxmmJcNsYWVP_Tp82iNB_ohrXhcjqWBy?usp=drive_link

In [25]:
import pandas as pd
df = pd.read_csv("C:\Users\Marcus\Downloads\dados.csv")
df

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (<ipython-input-25-5878f4bc7981>, line 2)

In [24]:
# treinamento de um modelo de arvore
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, classification_report

X = df.loc[:,['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']]
y = df.loc[:,'Class']
# Divisão em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Treinar um modelo simples (árvore de decisão)
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Previsões
y_pred = model.predict(X_test)

# Avaliação
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

KeyError: "None of [Index(['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11',\n       'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21',\n       'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount'],\n      dtype='object')] are in the [columns]"

In [ ]:
# de uma de uma rede neural
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, TensorDataset

# Configurações gerais
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
# Converter para tensores do PyTorch
X_train_tensor = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

# Criar DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Definir uma rede neural simples
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(29, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)  # Saída com 2 classes

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)  # Sem softmax, pois usaremos CrossEntropyLoss
        return x

# Inicializar a rede, função de perda e otimizador
model = SimpleNN().to(device)
criterion = nn.CrossEntropyLoss()  # Ignora o desbalanceamento por padrão
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Treinar a rede neural
epochs = 20
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # Forward
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward e otimização
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

# Avaliação do modelo
model.eval()
y_pred = []
y_true = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        y_pred.extend(predicted.cpu().numpy())
        y_true.extend(y_batch.cpu().numpy())

Epoch 1/20, Loss: 0.0150
Epoch 2/20, Loss: 0.0054
Epoch 3/20, Loss: 0.0049
Epoch 4/20, Loss: 0.0051
Epoch 5/20, Loss: 0.0049
Epoch 6/20, Loss: 0.0037
Epoch 7/20, Loss: 0.0036
Epoch 8/20, Loss: 0.0040
Epoch 9/20, Loss: 0.0033
Epoch 10/20, Loss: 0.0031
Epoch 11/20, Loss: 0.0032
Epoch 12/20, Loss: 0.0031
Epoch 13/20, Loss: 0.0029
Epoch 14/20, Loss: 0.0028
Epoch 15/20, Loss: 0.0030
Epoch 16/20, Loss: 0.0031
Epoch 17/20, Loss: 0.0027
Epoch 18/20, Loss: 0.0032
Epoch 19/20, Loss: 0.0027
Epoch 20/20, Loss: 0.0025


In [ ]:
# Avaliação
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9995318516437859


## 2.1 (3 pontos)
Ambos os modelos estão ótimos certo? Com alta acurácia...

Ou não? A acurácia de fato foi a melhor métrica utilizada para validar a perfomance do modelo? Caso não concorde qual a métrica na sua opnião deveria ser utilizada e justifique o motivo.

## 2.2 (2 pontos)
Utilizando o mesmo conjunto de dados, sem se preocupar com o pré-processamento de dados, é possível melhorar a perfomance do modelo pela métrica que voce acha que seria interessante?

Pode utilizar modelo de árvore ou a rede neural, alterar os hiperparametros, divisão no conjunto de treino, outros modelos que podem ser melhores, fique a vontade para escolher a solução que melhore a sua métrica de avaliação e justifique (mesmo que não tenha conseguido melhorar, oq acha que pode ter acontecido).